# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts**: le pides a Gemini que reescriba el código de tu figura y pegas su respuesta aquí.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

**Escriba los nombres de los integrantes aca.**
______________________________________________________________________  
 1. Sara Serpa 202526509  
    


## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Ejercicio 1
ventas_producto = df.groupby("producto")["total"].sum()
ordenado = ventas_producto.sort_values(ascending=False)
top_productos = ordenado.head(5)
top_productos.plot.barh(title="Top 5 productos por ingresos")
plt.xlabel("Ingresos ($)")
plt.show()

In [ ]:
# ✅ Verificación
assert len(top_productos) == 5, "Debe haber exactamente 5 productos"
assert top_productos.index[0] == "Cuaderno", "El producto con más ingresos no coincide"
assert abs(top_productos.iloc[0] - 20712345) < 1, "Revisa que estés sumando la columna total"
print("Ejercicio 1 correcto")

### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Ejercicio 2
dos_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = dos_ciudades.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")
ing_mes_ciudad.plot.line(marker="o", title="Ingresos mensuales: Bogotá vs Medellín")
plt.ylabel("Ingresos ($)")
plt.show()

In [ ]:
# ✅ Verificación
assert ing_mes_ciudad.shape == (12, 2), f"Se esperaba una tabla de 12 meses x 2 ciudades, tienes {ing_mes_ciudad.shape}"
assert set(ing_mes_ciudad.columns) == {"Bogotá", "Medellín"}, "Las columnas deben ser las dos ciudades"
assert abs(ing_mes_ciudad["Bogotá"].sum() - 37847614) < 1, "El total de Bogotá no coincide"
print("Ejercicio 2 correcto ✔")

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Ejercicio 3
ing_canal = df.groupby("canal")["total"].sum()
ing_canal.plot.pie(autopct="%.1f%%", title="Participación de los ingresos por canal")
plt.ylabel("")
plt.show()

In [ ]:
# ✅ Verificación
assert abs(ing_canal.sum() - 118668012) < 1, "La suma de los canales debe ser el ingreso total"
assert abs(ing_canal["Telefónico"] / ing_canal.sum() - 0.520) < 0.01, "La participación del canal Telefónico no coincide"
print("Ejercicio 3 correcto ✔")

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Ejercicio 4
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(stacked=True, title="Categorías que compra cada segmento (%)")
plt.ylabel("% de las ventas")
plt.show()

In [ ]:
# ✅ Verificación
assert tabla_seg_cat.shape == (3, 5), f"Se esperaba 3 segmentos x 5 categorías, tienes {tabla_seg_cat.shape}"
assert (abs(tabla_seg_cat.sum(axis=1) - 100) < 0.01).all(), "Cada fila debe sumar 100 (usa normalize y multiplica por 100)"
print("Ejercicio 4 correcto ✔")

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Ejercicio 5
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("total", "count"), ticket_prom=("total", "mean"))
vend = vend.reset_index()

for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    plt.scatter(datos["n_ventas"], datos["ticket_prom"], label=ciudad)

plt.title("Productividad de los vendedores")
plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio ($)")
plt.legend()
plt.show()

In [ ]:
# ✅ Verificación
assert vend.shape[0] == 12, f"Debe haber una fila por vendedor (12), tienes {vend.shape[0]}"
assert {"vendedor", "ciudad", "n_ventas", "ticket_prom"} <= set(vend.columns), "Faltan columnas: vendedor, ciudad, n_ventas, ticket_prom"
assert vend["n_ventas"].sum() == 2000, "La suma de n_ventas debe ser 2000"
print("Ejercicio 5 correcto ✔")

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Ejercicio 6
sem2 = df[df["mes"] >= 7]
sem2.boxplot(column="total", by="segmento", showfliers=False)
plt.title("Valor de las ventas por segmento (segundo semestre)")
plt.suptitle("")
plt.ylabel("Total de la venta ($)")
plt.show()

In [ ]:
# ✅ Verificación
assert len(sem2) == 1050, f"El segundo semestre debe tener 1050 ventas, tienes {len(sem2)}"
print("Ejercicio 6 correcto ✔")

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Ejercicio 7: código de la figura 2x3
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
dos_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = dos_ciudades.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")
ing_canal = df.groupby("canal")["total"].sum()
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("total", "count"), ticket_prom=("total", "mean")).reset_index()
sem2 = df[df["mes"] >= 7]

fig, axs = plt.subplots(2, 3, figsize=(18, 10))

top_productos.plot.barh(ax=axs[0, 0], title="Top 5 productos por ingresos")

ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1], title="Ingresos mensuales: Bogotá vs Medellín")

ing_canal.plot.pie(autopct="%.1f%%", ax=axs[0, 2], title="Ingresos por canal")
axs[0, 2].set_ylabel("")

tabla_seg_cat.plot.bar(stacked=True, ax=axs[1, 0], title="Categorías por segmento (%)")

for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(datos["n_ventas"], datos["ticket_prom"], label=ciudad)
axs[1, 1].set_title("Productividad de los vendedores")
axs[1, 1].set_xlabel("Número de ventas")
axs[1, 1].set_ylabel("Ticket promedio ($)")
axs[1, 1].legend()

sem2.boxplot(column="total", by="segmento", showfliers=False, ax=axs[1, 2])
axs[1, 2].set_title("Ventas por segmento (2.º semestre)")

fig.suptitle("Ventas de papelerías 2025")
plt.tight_layout()

In [ ]:
%run -i figura_base.py

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt** y se lo envías a Gemini (en [gemini.google.com](https://gemini.google.com) o la IA que prefieras). Gemini **reescribe el código** de tu figura para que se vea profesional, casi como un dashboard. Tú pegas ese código en el notebook y lo ejecutas.

**El ciclo, cada vez:**
1. Ejecuta la celda de herramientas: te muestra **tu código actual** y las **reglas técnicas**.
2. Escribe tu prompt (Ejercicios 8 y 9).
3. En el chat de la IA pega, en este orden: **tu prompt + tu código + las reglas**.
4. Copia **solo el código** de la respuesta (botón *copiar* del bloque) y pégalo en la celda `%%writefile figura_v1.py`. **No incluyas las líneas con \`\`\`** ni el texto que la IA escriba fuera del bloque.
5. Ejecuta la celda y luego `mostrar("v1")` para ver la figura.

### Herramientas *(ya están hechas)*
Ejecuta la celda: define la función `mostrar(nombre)` (dibuja la versión guardada en `figura_<nombre>.py`) e imprime lo que debes copiar a la IA.

In [ ]:
REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual; cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def mostrar(nombre):
    """Dibuja la versión guardada en figura_<nombre>.py."""
    plt.close("all")
    codigo = open(f"figura_{nombre}.py", encoding="utf-8").read().replace("plt.show()", "")
    exec(codigo, globals())
    display(fig)
    plt.close(fig)


print("=== 1) COPIA ESTE CÓDIGO (tu figura actual) ===")
print(open("figura_base.py", encoding="utf-8").read())
print("\n=== 2) COPIA ESTAS REGLAS al final de tu prompt ===")
print(REGLAS_FIGURA)

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

Ejecuta la primera celda para imprimir tu prompt y cópialo a la IA (con tu código y las reglas). Luego pega el código que te devuelva en la segunda celda y muestra la figura con la tercera.

In [ ]:
prompt_v1 = """
Reescribe el código de esta figura de matplotlib para que parezca un dashboard profesional.
PÚBLICO Y OBJETIVO: la gerente general y la junta directiva de una cadena de papelerías. La figura se presenta en una junta para decidir inventario, canales de venta y metas por ciudad, así que debe entenderse en pocos segundos.
PALETA Y TIPOGRAFÍA: fondo blanco. Usa azul oscuro (#1f4e79) para Bogotá y para el dato más importante de cada gráfico, naranja (#f28e2b) para Medellín y gris (#9e9e9e) para lo secundario. Usa el mismo color para cada ciudad en todos los gráficos. Títulos de los gráficos en negrita, tamaño 13; título general tamaño 20; resto del texto tamaño 11.
TÍTULOS: cada gráfico debe tener un título que diga la conclusión, no solo el nombre. Por ejemplo: "Cuaderno lidera los ingresos", "Bogotá supera a Medellín en 9 de 12 meses", "El canal Telefónico aporta más de la mitad". Título general: "Dashboard de ventas · Papelerías 2025".
EJES Y NÚMEROS: los ingresos en millones con el formato "$ 12,3 M"; los valores en pesos con separador de miles ("$ 60.000"). Los meses como Ene, Feb, Mar... en lugar de 1, 2, 3. Etiquetas de ejes claras y en español.
LIMPIEZA VISUAL: quita los bordes de arriba y de la derecha, usa una cuadrícula suave y gris solo en un eje, pon el valor sobre cada barra del top 5, saca la leyenda de las barras apiladas fuera del gráfico y quita el título automático que pone el boxplot.
"""

print(prompt_v1)    # cópialo en la IA, junto con tu código y las reglas

In [ ]:
%%writefile figura_v1.py
import numpy as np
import matplotlib.ticker as mticker

# ----- Datos: los mismos 6 gráficos -----
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
dos_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = dos_ciudades.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")
ing_canal = df.groupby("canal")["total"].sum()
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("total", "count"), ticket_prom=("total", "mean")).reset_index()
sem2 = df[df["mes"] >= 7]

# ----- Estilo -----
AZUL = "#1f4e79"
NARANJA = "#f28e2b"
GRIS = "#9e9e9e"
COLOR_CIUDAD = {"Bogotá": AZUL, "Medellín": NARANJA, "Cali": "#59a14f",
                "Barranquilla": "#e15759", "Bucaramanga": "#b07aa1"}
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold"})

def millones(x, pos):
    return "$ " + format(x / 1e6, ".1f").replace(".", ",") + " M"

def pesos(x, pos):
    return "$ " + format(x, ",.0f").replace(",", ".")

fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor("white")

# 1) Top 5 productos
ax = axs[0, 0]
datos = top_productos.sort_values()
colores = [GRIS] * (len(datos) - 1) + [AZUL]
ax.barh(datos.index, datos.values, color=colores)
for i, v in enumerate(datos.values):
    ax.text(v, i, "  " + millones(v, None), va="center", fontsize=10)
ax.set_xlim(0, datos.max() * 1.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(millones))
ax.set_title("Cuaderno lidera los ingresos")
ax.set_xlabel("Ingresos")

# 2) Bogotá vs Medellín
ax = axs[0, 1]
for ciudad in ["Bogotá", "Medellín"]:
    ax.plot(ing_mes_ciudad.index, ing_mes_ciudad[ciudad], marker="o", linewidth=2.2,
            color=COLOR_CIUDAD[ciudad], label=ciudad)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(millones))
ax.set_title("Bogotá supera a Medellín en 9 de 12 meses")
ax.set_ylabel("Ingresos")
ax.legend(frameon=False)

# 3) Canales
ax = axs[0, 2]
ax.pie(ing_canal.values, labels=ing_canal.index, autopct="%.1f%%", startangle=90,
       colors=[NARANJA, AZUL, GRIS], wedgeprops={"edgecolor": "white", "linewidth": 2},
       textprops={"fontsize": 11})
ax.set_title("El canal Telefónico aporta más de la mitad")

# 4) Segmento x categoría
ax = axs[1, 0]
tabla_seg_cat.plot.bar(stacked=True, ax=ax, width=0.65,
                       color=["#b07aa1", "#e15759", AZUL, NARANJA, "#59a14f"])
ax.set_title("Corporativos compran Papel; los demás, Escritura")
ax.set_xlabel("")
ax.set_ylabel("% de las ventas")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Categoría", frameon=False, bbox_to_anchor=(1.0, 1.0), loc="upper left")

# 5) Vendedores
ax = axs[1, 1]
for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    ax.scatter(datos["n_ventas"], datos["ticket_prom"], s=90, color=COLOR_CIUDAD[ciudad],
               label=ciudad, edgecolor="white")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pesos))
ax.set_title("Laura Gómez vende más; Paula Vargas, más caro")
ax.set_xlabel("Número de ventas")
ax.set_ylabel("Ticket promedio")
ax.legend(frameon=False, fontsize=9)

# 6) Boxplot segundo semestre
ax = axs[1, 2]
sem2.boxplot(column="total", by="segmento", showfliers=False, ax=ax, grid=False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pesos))
ax.set_title("Mayoristas y corporativos hacen las ventas grandes")
ax.set_xlabel("")
ax.set_ylabel("Total de la venta")

# ----- Limpieza -----
for ax in axs.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axs[0, 0].grid(axis="x", alpha=0.3)
for ax in [axs[0, 1], axs[1, 0], axs[1, 1], axs[1, 2]]:
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle("Dashboard de ventas · Papelerías 2025", fontsize=20, fontweight="bold")
plt.tight_layout()

In [ ]:
mostrar("v1")

### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (otra paleta, letras más grandes, una dona en lugar del pie, anotar el valor máximo del gráfico de líneas…). Es la **iteración de prompts** que vimos en clase. Repite el ciclo: copia `prompt_v2` a la IA (con tu código base y las reglas), pega el código en `figura_v2.py` y muestra la figura.

In [ ]:
prompt_v2 = prompt_v1 + """

AJUSTES:
1. Cambia el gráfico de pie por una dona y escribe en el centro el ingreso total de la cadena ("$ 118,7 M").
2. En el gráfico de líneas marca con una anotación el mes de mayor ingreso de Bogotá (enero, $ 4,8 M).
3. En las barras apiladas escribe el porcentaje dentro de cada parte, solo si es mayor que 8 %.
4. En el boxplot pinta las cajas con colores suaves (no blancas) y la mediana en negro.
5. Agrega debajo del título general un subtítulo gris: "2.000 ventas · enero a diciembre de 2025".
"""

print(prompt_v2)

In [ ]:
%%writefile figura_v2.py
import numpy as np
import matplotlib.ticker as mticker

# ----- Datos: los mismos 6 gráficos -----
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
dos_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = dos_ciudades.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")
ing_canal = df.groupby("canal")["total"].sum()
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("total", "count"), ticket_prom=("total", "mean")).reset_index()
sem2 = df[df["mes"] >= 7]

# ----- Estilo -----
AZUL = "#1f4e79"
NARANJA = "#f28e2b"
GRIS = "#9e9e9e"
COLOR_CIUDAD = {"Bogotá": AZUL, "Medellín": NARANJA, "Cali": "#59a14f",
                "Barranquilla": "#e15759", "Bucaramanga": "#b07aa1"}
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold"})

def millones(x, pos):
    return "$ " + format(x / 1e6, ".1f").replace(".", ",") + " M"

def pesos(x, pos):
    return "$ " + format(x, ",.0f").replace(",", ".")

fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor("white")

# 1) Top 5 productos
ax = axs[0, 0]
datos = top_productos.sort_values()
colores = [GRIS] * (len(datos) - 1) + [AZUL]
ax.barh(datos.index, datos.values, color=colores)
for i, v in enumerate(datos.values):
    ax.text(v, i, "  " + millones(v, None), va="center", fontsize=10)
ax.set_xlim(0, datos.max() * 1.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(millones))
ax.set_title("Cuaderno lidera los ingresos")
ax.set_xlabel("Ingresos")

# 2) Bogotá vs Medellín
ax = axs[0, 1]
for ciudad in ["Bogotá", "Medellín"]:
    ax.plot(ing_mes_ciudad.index, ing_mes_ciudad[ciudad], marker="o", linewidth=2.2,
            color=COLOR_CIUDAD[ciudad], label=ciudad)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(millones))
ax.set_title("Bogotá supera a Medellín en 9 de 12 meses")
ax.set_ylabel("Ingresos")
mes_max = ing_mes_ciudad["Bogotá"].idxmax()
valor_max = ing_mes_ciudad["Bogotá"].max()
ax.annotate("Máximo: " + millones(valor_max, None), xy=(mes_max, valor_max),
            xytext=(mes_max + 1.5, valor_max * 1.05), fontsize=10, color=AZUL,
            arrowprops={"arrowstyle": "->", "color": AZUL})
ax.set_ylim(0, valor_max * 1.2)
ax.legend(frameon=False, loc="lower left")

# 3) Canales
ax = axs[0, 2]
ax.pie(ing_canal.values, labels=ing_canal.index, autopct="%.1f%%", startangle=90,
       colors=[NARANJA, AZUL, GRIS], pctdistance=0.78,
       wedgeprops={"edgecolor": "white", "linewidth": 2, "width": 0.42},
       textprops={"fontsize": 11})
ax.text(0, 0.08, millones(ing_canal.sum(), None), ha="center", fontsize=16, fontweight="bold")
ax.text(0, -0.15, "ingreso total", ha="center", fontsize=10, color="#555555")
ax.set_title("El canal Telefónico aporta más de la mitad")

# 4) Segmento x categoría
ax = axs[1, 0]
tabla_seg_cat.plot.bar(stacked=True, ax=ax, width=0.65,
                       color=["#b07aa1", "#e15759", AZUL, NARANJA, "#59a14f"])
ax.set_title("Corporativos compran Papel; los demás, Escritura")
ax.set_xlabel("")
ax.set_ylabel("% de las ventas")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Categoría", frameon=False, bbox_to_anchor=(1.0, 1.0), loc="upper left")
for barras in ax.containers:
    etiquetas = []
    for b in barras:
        if b.get_height() > 8:
            etiquetas.append(format(b.get_height(), ".0f") + " %")
        else:
            etiquetas.append("")
    ax.bar_label(barras, labels=etiquetas, label_type="center", color="white", fontsize=9)

# 5) Vendedores
ax = axs[1, 1]
for ciudad in vend["ciudad"].unique():
    datos = vend[vend["ciudad"] == ciudad]
    ax.scatter(datos["n_ventas"], datos["ticket_prom"], s=90, color=COLOR_CIUDAD[ciudad],
               label=ciudad, edgecolor="white")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pesos))
ax.set_title("Laura Gómez vende más; Paula Vargas, más caro")
ax.set_xlabel("Número de ventas")
ax.set_ylabel("Ticket promedio")
ax.legend(frameon=False, fontsize=9)

# 6) Boxplot segundo semestre
ax = axs[1, 2]
cajas = sem2.boxplot(column="total", by="segmento", showfliers=False, ax=ax, grid=False,
                     patch_artist=True, return_type="dict",
                     medianprops={"color": "black", "linewidth": 2})
for caja, color in zip(cajas["total"]["boxes"], ["#c6dbef", "#fdd0a2", "#d9d9d9"]):
    caja.set_facecolor(color)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pesos))
ax.set_title("Mayoristas y corporativos hacen las ventas grandes")
ax.set_xlabel("")
ax.set_ylabel("Total de la venta")

# ----- Limpieza -----
for ax in axs.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axs[0, 0].grid(axis="x", alpha=0.3)
for ax in [axs[0, 1], axs[1, 0], axs[1, 1], axs[1, 2]]:
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle("Dashboard de ventas · Papelerías 2025", fontsize=20, fontweight="bold", y=1.0)
fig.text(0.5, 0.955, "2.000 ventas · enero a diciembre de 2025", ha="center", fontsize=12, color="#666666")
plt.tight_layout(rect=(0, 0, 1, 0.95))

In [ ]:
mostrar("v2")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`), repite el ciclo con `figura_v3.py` y muestra `mostrar("v3")`.

In [ ]:
# prompt_v3 = prompt_v2 + """
#
# AJUSTES FINALES: ...
# """
# print(prompt_v3)

In [ ]:
# ✅ Verificación
import os
assert os.path.exists("figura_v1.py") and os.path.exists("figura_v2.py"), "Faltan figura_v1.py y figura_v2.py: pega el código de cada versión"
assert "ESCRIBE AQUÍ" not in prompt_v1 and prompt_v2 != prompt_v1 and "ESCRIBE AQUÍ" not in prompt_v2, "Completa tus prompts v1 y v2"
print("Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v2"      # cambia por "v1", "v2" o "v3"

hallazgos = [
    "Bogotá genera el 31,9 % de los ingresos de la cadena ($ 37,8 M de $ 118,7 M).",
    "El canal Telefónico concentra el 51,8 % de los ingresos, casi el triple que el canal Online (19,3 %).",
    "Cuaderno es el producto con más ingresos ($ 20,7 M), $ 6,4 M más que Block de notas, el segundo.",
]

In [ ]:
# ✅ Verificación
import os
assert os.path.exists(f"figura_{figura_elegida}.py"), f"No existe figura_{figura_elegida}.py: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
mostrar(figura_elegida)

print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, todos los integrantes deben estar como colaboradores en el repositorio, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook) (3) El notebook (ipynb) con los nombres de todos los integrantes.

### Qué corregimos de lo que hizo la IA

*(IA usada: Gemini)*

1. En la versión 1 el pie se veía plano y no mostraba el total de la cadena; en el prompt_v2 pedimos una dona con el ingreso total (118,7 M) escrito en el centro.

2. En las barras apiladas había que adivinar los porcentajes mirando el eje; pedi escribir el % dentro de cada parte, pero solo si era mayor que 8 % para que no se amontonaran los números.

3. El boxplot salía con cajas blancas que casi no se veían y el gráfico de líneas no destacaba el mes más alto; pedi colores suaves con la mediana en negro y una anotación en el máximo de Bogotá (enero, 4,8 M).